# Imports

In [ ]:
import json
import os
import pickle
import warnings
from ast import literal_eval

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal as signal
import uproot
from itables import init_notebook_mode
from matplotlib.colors import LogNorm
from matplotlib.ticker import AutoMinorLocator, MaxNLocator, ScalarFormatter
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage.measure import LineModelND, ransac
from sklearn.cluster import DBSCAN
from sklearn.linear_model import RANSACRegressor
from skspatial.objects import Cylinder, Line, Plane, Point, Triangle
from tqdm.auto import tqdm

init_notebook_mode(all_interactive=True)

# Parameters

README 

- `non_track_keys = 2` has to be updated if new keys are added to an event in metrics

In [ ]:
# File paths
path_prefix = "./MC"
file_label = "MC"

charge_file = "../Monte Carlo/singlecube_cosmics_hits.root"

In [ ]:
# Load options
reload_files = True

# Save options
overwrite_metrics = True
save_figures = True

# Plotting options
flip_x = True
individual_plots = np.arange(1, 101, 1)
show_figures = True

# Events to process
event_list = None

# Noisy Pixels
channel_disable_list = [7]

# Units for plot labels
q_unit = "e"  # After applying charge_gain
xy_unit = "mm"
z_unit = "mm"
dh_unit = "?" if z_unit != xy_unit else xy_unit
time_unit = "ns"

# Conversion factors
charge_gain = 245  # mV to e
quadrant_size = 32  # One SiPM + LArPix cell
detector_z = 300
detector_x = quadrant_size * 8
detector_y = quadrant_size * 8
sipm_size = 6
pixel_size = 3
pixel_pitch = 4

# Transform for real readout
simulate_dead_area = True
x_translation = -pixel_size / 2  # + quadrant_size * 2
y_translation = pixel_size / 2  # + quadrant_size * 1.5
z_translation = -detector_z / 2

# DBSCAN parameters for charge clustering
min_samples = 2
xy_epsilon = 8  # 8 ideal
z_epsilon = 8  # 8 ideal

# RANSAC parameters for line fitting
ransac_residual_threshold = 6  # 6 ideal for charge, 35 ideal for light
ransac_max_trials = 1000
ransac_min_samples = 2  # 2 ideal for charge, 3 ideal for light

# Force parameters for cylinder
force_dh = None
force_dr = None

# Filters for post processing
score_cutoff = -1.0
max_score = 1.0
min_track_length = 30
max_track_length = np.inf
max_tracks = 1

# Other
non_track_keys = 5

# Functions

## Parameter calculators

In [ ]:
# Cylinder parameters for dQ/dx
def get_dh(unit_vector):
    if force_dh is not None:
        return force_dh

    dl_vector = np.array([xy_epsilon, xy_epsilon, z_epsilon])
    min_dh = np.linalg.norm(dl_vector) / 4
    max_dh = 2 * np.linalg.norm(dl_vector)
    dl_projection = abs(np.dot(unit_vector, dl_vector))
    dh = min(max(dl_projection, min_dh), max_dh)

    return dh


def get_dr(rmse):
    if force_dr is not None:
        return force_dr

    dl_vector = np.array([xy_epsilon, xy_epsilon, z_epsilon])
    min_dr = np.linalg.norm(dl_vector) / 4
    dr = max(rmse, min_dr)

    return dr

## Data handling

### Charge

In [ ]:
# Create a list of fake data
def generate_dead_area(z_range, buffer=1, disable_coords=[(1, 2)], first_chip=(2, 1)):
    # Dead area on SiPMs
    fake_x = []
    fake_y = []
    fake_z = []

    for k in range((detector_x // quadrant_size)):
        for l in range((detector_y // quadrant_size)):
            # Dead area on chips 33, 44, 54, 64
            if (l + first_chip[0], k + first_chip[1]) in [
                (3, 3),
                (4, 4),
                (5, 4),
                (6, 4),
            ]:
                # continue
                temp_x, temp_y, temp_z = np.meshgrid(
                    np.linspace(pixel_pitch, quadrant_size - pixel_pitch, 6)
                    - detector_x / 2
                    + quadrant_size * (k),
                    -np.linspace(pixel_pitch, quadrant_size - pixel_pitch, 6)
                    + detector_y / 2
                    - quadrant_size * (l),
                    z_range,
                )

                fake_x.extend(temp_x.flatten())
                fake_y.extend(temp_y.flatten())
                fake_z.extend(temp_z.flatten())
            # Dead area on chip 42
            elif k + first_chip[1] == 2 and l + first_chip[0] == 4:
                temp_x, temp_y, temp_z = np.meshgrid(
                    np.linspace(pixel_pitch, quadrant_size - pixel_pitch, 6),
                    np.linspace(pixel_pitch, quadrant_size - pixel_pitch, 6),
                    z_range,
                )

                mask = (temp_y - quadrant_size / 2) - (temp_x - quadrant_size / 2) >= 0
                temp_x = temp_x[mask] - detector_x / 2 + quadrant_size * (k)
                temp_y = -temp_y[mask] + detector_y / 2 - quadrant_size * (l)
                temp_z = temp_z[mask]

                temp_x2, temp_y2, temp_z2 = np.meshgrid(
                    [x1 + sipm_size / 2 - buffer + quadrant_size * k],
                    [y1 + sipm_size / 2 - buffer - quadrant_size * l],
                    z_range,
                )

                fake_x.extend(np.append(temp_x, temp_x2))
                fake_y.extend(np.append(temp_y, temp_y2))
                fake_z.extend(np.append(temp_z, temp_z2))
            # Dead area on SiPMs
            else:
                x1 = -detector_x / 2 + quadrant_size / 2
                y1 = detector_y / 2 - quadrant_size / 2

                temp_x, temp_y, temp_z = np.meshgrid(
                    np.array([x1 - sipm_size / 2 + buffer, x1 + sipm_size / 2 - buffer])
                    + quadrant_size * k,
                    np.array([y1 + sipm_size / 2 - buffer, y1 - sipm_size / 2 + buffer])
                    - quadrant_size * l,
                    z_range,
                )

                # Removing channel 7
                disable_x, disable_y, disable_z = [], [], []

                for coords in disable_coords:
                    temp_x2, temp_y2, temp_z2 = np.meshgrid(
                        np.array(
                            [
                                -detector_x / 2
                                + (coords[0] * pixel_pitch)
                                + (pixel_pitch - pixel_size)
                            ]
                        )
                        + quadrant_size * k,
                        np.array(
                            [
                                detector_y / 2
                                - (coords[1] * pixel_pitch)
                                - (pixel_pitch - pixel_size)
                            ]
                        )
                        - quadrant_size * l,
                        z_range,
                    )
                    disable_x.extend(temp_x2)
                    disable_y.extend(temp_y2)
                    disable_z.extend(temp_z2)

                fake_x.extend(np.append(temp_x, disable_x))
                fake_y.extend(np.append(temp_y, disable_y))
                fake_z.extend(np.append(temp_z, disable_z))

    fake_x = np.array(fake_x)
    fake_y = np.array(fake_y)
    fake_z = np.array(fake_z)

    fake_data = np.c_[np.power(-1, flip_x) * fake_x, fake_y, fake_z]

    return fake_data

In [ ]:
# Apply DBSCAN clustering
def cluster_hits(hitArray):
    if simulate_dead_area:
        # First stage clustering
        z_intervals = []
        first_stage = DBSCAN(eps=xy_epsilon, min_samples=min_samples).fit(
            hitArray[:, 0:2]
        )
        for label in first_stage.labels_:
            if label > -1:
                mask = first_stage.labels_ == label
                z = hitArray[mask, 2]
                z_intervals.append((min(z), max(z)))

        # Sort the intervals based on their start points
        sorted_intervals = sorted(z_intervals, key=lambda interval: interval[0])

        # Initialize a list to store the intervals representing the empty space
        empty_space_ranges = []
        # Iterate through the sorted intervals to find the gaps
        for i in range(len(sorted_intervals) - 1):
            current_interval = sorted_intervals[i]
            next_interval = sorted_intervals[i + 1]

            # Calculate the gap between the current interval and the next interval
            gap_start = current_interval[1]
            gap_end = next_interval[0]

            # Check if there is a gap (empty space) between intervals
            if gap_end > gap_start and gap_end < gap_start + 40:
                empty_space_ranges.append(np.arange(gap_start, gap_end, z_epsilon))

        if not empty_space_ranges:
            empty_space_ranges.append(
                np.arange(
                    np.mean(hitArray[:, 2]) - np.std(hitArray[:, 2]),
                    np.mean(hitArray[:, 2]) + np.std(hitArray[:, 2]),
                    z_epsilon,
                )
            )

        z_range = np.concatenate(empty_space_ranges)

        # Create a list of holes
        fake_data = generate_dead_area(z_range)
        fake_data_count = len(fake_data)

        # Second stage clustering
        # Combine fake to true data
        second_stage_data = np.concatenate([hitArray, fake_data])

    else:
        second_stage_data = hitArray

    second_stage = DBSCAN(eps=xy_epsilon, min_samples=1).fit(second_stage_data[:, 0:2])

    # Third stage clustering
    # Create a new array with z and labels
    third_stage_z = np.c_[second_stage.labels_ * 1e3, second_stage_data[:, 2]]
    flag = second_stage.labels_ > -1
    third_stage_data = third_stage_z[flag].copy()
    third_stage = DBSCAN(
        eps=z_epsilon, min_samples=min_samples, metric="chebyshev"
    ).fit(third_stage_data)

    if simulate_dead_area:
        # Remove fake data
        # Shift labels by 1 so that negative values are reserved for outliers
        labels = third_stage.labels_[:-fake_data_count] + 1
    else:
        labels = third_stage.labels_ + 1

    return labels


# Apply Ransac Fit
def ransacFit(
    hitArray,
    weightArray=None,
    min_samples=None,
    residual_threshold=None,
):
    if weightArray is not None:
        estimator = RANSACRegressor(
            min_samples=min_samples,
            max_trials=ransac_max_trials,
            residual_threshold=residual_threshold,
        )
        last_column = len(hitArray[0]) - 1
        inliers = estimator.fit(
            hitArray[:, 0:last_column],
            hitArray[:, last_column],
            sample_weight=weightArray,
        ).inlier_mask_

        # Check it enouth inliers
        if sum(inliers) > ransac_min_samples:
            score = estimator.score(
                hitArray[:, 0:last_column], hitArray[:, last_column]
            )
        else:
            score = np.nan
    else:
        model_robust, inliers = ransac(
            hitArray,
            LineModelND,
            min_samples=min_samples,
            residual_threshold=residual_threshold,
            max_trials=ransac_max_trials,
        )

        # Check it enouth inliers
        if sum(inliers) > ransac_min_samples:
            score = model_robust.residuals(hitArray)
        else:
            score = np.nan

    outliers = inliers == False
    return inliers, outliers, score


# Apply best line fit
def lineFit(hitArray):
    line = Line.best_fit(hitArray)
    max_point = Point(np.max(hitArray, axis=0))
    min_point = Point(np.min(hitArray, axis=0))
    centre_point = (max_point + min_point) / 2
    line.point = line.project_point(centre_point)
    residuals = np.array([line.distance_point(point) for point in hitArray])

    # Calculate chi-squared
    mse = np.sum(residuals**2) / len(residuals)

    return line, mse


# Calculate dQ/dx from a line fit
def dqdx(hitArray, q, line_fit, dh, dr, h, ax=None):
    # Cylinder steps for dQ/dx
    steps = np.arange(-3 * dh, h + 3 * dh, dh)

    # Mask of points that have been accounted for
    counted = np.zeros(len(q), dtype=bool)

    # Array of dQ values for each step
    dq_i = np.zeros(len(steps), dtype=float)

    for step_idx, step in enumerate(steps):
        cylinder_fit = Cylinder(
            line_fit.to_point(h / 2 - step),
            -line_fit.direction.unit() * dh,
            dr,
        )
        if ax is not None:
            cylinder_fit.plot_3d(ax)

        for point_idx, point in enumerate(hitArray):
            if not counted[point_idx] and cylinder_fit.is_point_within(point):
                counted[point_idx] = True
                dq_i[step_idx] += q[point_idx]

    return dq_i


# Fit clusters with Ransac method
def fit_hit_clusters(
    hitArray, q, labels, ax2d=None, ax3d=None, plot_cyl=False, refit_outliers=True
):
    metrics = {}
    # Fit clusters
    idx = 0
    condition = lambda: idx < len(np.unique(labels))
    while condition():
        label = np.unique(labels)[idx]
        mask = labels == label
        if label > 0 and mask.sum() > min_samples:
            xyz_c = hitArray[mask]
            q_c = np.array(q)[mask]

            norm = np.linalg.norm(np.max(xyz_c, axis=0) - np.min(xyz_c, axis=0))

            # Fit the model
            inliers, outliers, score = ransacFit(
                xyz_c,
                weightArray=q_c - min(q_c) + 1,
                min_samples=ransac_min_samples,
                residual_threshold=ransac_residual_threshold,
            )

            # Refit outliers
            level_1 = np.where(mask)[0]
            level_2 = np.where(outliers)[0]
            level_3 = level_1[level_2]

            if refit_outliers and sum(outliers) > min_samples:
                outlier_labels = cluster_hits(xyz_c[outliers])
                last_label = max(labels) + 1
                # Assign positive labels to clustered outliers and negative labels to unlclustered outliers
                for i, j in enumerate(level_3):
                    labels[j] = (outlier_labels[i] + last_label) * (
                        1 if outlier_labels[i] > 0 else -1
                    )
            else:
                # Assign negative labels to outliers
                for j in level_3:
                    labels[j] = -labels[j]

            if sum(inliers) > min_samples:
                line_fit, mse = lineFit(xyz_c[inliers])

                if ax2d is not None:
                    # 2D plot
                    line_fit.plot_2d(
                        ax2d,
                        t_1=-norm / 2,
                        t_2=norm / 2,
                        c="red",
                        label=f"Track {label}",
                        zorder=10,
                    )
                if ax3d is not None:
                    # 3D plot
                    line_fit.plot_3d(
                        ax3d,
                        t_1=-norm / 2,
                        t_2=norm / 2,
                        c="red",
                        label=f"Track {label}",
                    )

                # Calculate dQ/dx
                dh = get_dh(line_fit.direction)
                dr = get_dr(np.sqrt(mse))

                dq_i = dqdx(
                    xyz_c[inliers],
                    q_c[inliers],
                    line_fit,
                    dh=dh,
                    dr=dr,
                    h=norm,
                    ax=ax3d if ax3d is not None and plot_cyl else None,
                )

                q_eff = dq_i.sum() / q_c[inliers].sum()
                if dq_i.sum() != 0:
                    dq = dq_i
                else:
                    dq = 0

                metrics[label] = {
                    "Fit_line": line_fit,
                    "Fit_norm": norm,
                    "Fit_mse": mse,
                    "RANSAC_score": score,
                    "q_eff": q_eff,
                    "dQ": dq,
                    "dx": dh,
                }

        idx = np.unique(labels).tolist().index(label) + 1

    metrics["Fit_labels"] = labels

    return metrics

### Helpers

In [ ]:
def max_std(array, ax=None, array_max=None, min_count_ratio=0.9, max_std_ratio=0.5):
    max_std = array.std()
    max_count = len(array)
    if array_max is None:
        array_max = np.percentile(array, min(min_count_ratio * 100 + 1, 100))

    std = []
    count = []
    x_range = range(int(min(array)), (int(array_max) + 1), 1)
    for i in x_range:
        cut = array[array < i]
        std.append(cut.std())
        count.append(len(cut))

    std = np.array(std)
    count = np.array(count)
    condition = ((count / max_count).round(3) >= min_count_ratio) & (
        (std / max_std).round(3) <= max_std_ratio
    )
    vline = x_range[
        (
            np.where(condition)[0][-1]
            if np.any(condition)
            else (count / max_count > min_count_ratio).argmax()
        )
    ]

    print(
        "Max STD ratio",
        max_std_ratio,
        "limited to",
        min_count_ratio * 100,
        "% of events:",
        vline,
        "\n",
    )
    if ax is not None:
        ax.plot(std / max_std, label="STD ratio")
        ax.plot(count / max_count, label="Event count ratio")
        ax.axvline(vline, ls="--", c="r", label=f"{min_count_ratio*100}% of events")
        ax.legend()
        ax.tick_params(
            axis="both", direction="inout", which="major", top=True, right=True
        )
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.grid(alpha=0.25)

    return vline

In [ ]:
def cluster_hot_bins(min_n_ratio, n, x_edges, y_edges, scale=(1, 1), eps=8):
    n[np.isnan(n)] = 0
    min_n = min_n_ratio * n.max()
    filtered_n2 = n[n >= min_n]
    bin_centers_x = 0.5 * (x_edges[1:] + x_edges[:-1])
    bin_centers_y = 0.5 * (y_edges[1:] + y_edges[:-1])
    filtered_centers_x, filtered_centers_y = np.array(
        np.meshgrid(bin_centers_x, bin_centers_y)
    )
    filtered_centers_x = filtered_centers_x[n.T > min_n]
    filtered_centers_y = filtered_centers_y[n.T > min_n]
    dbscan = DBSCAN(
        eps=eps, min_samples=int(np.sqrt(len(filtered_centers_y))), metric="chebyshev"
    ).fit(np.c_[filtered_centers_x / scale[0], filtered_centers_y / scale[1]])
    return filtered_centers_x, filtered_centers_y, dbscan.labels_

In [ ]:
# Peak finding algorithm for integration
def integrate_peaks(waveform, buffer_size=10, height=0.1, prominence=0.05):
    # Find peaks in the filtered waveform
    peaks, properties = signal.find_peaks(
        waveform, height=height, prominence=prominence
    )

    integration_result = 0
    start_index = 0  # Initialize the start index
    end_index = 0  # Initialize the end index

    for peak in peaks:
        # Determine the potential start and end indices
        potential_start = max(0, peak - buffer_size)
        potential_end = min(len(waveform), peak + buffer_size)

        # If the potential start is within the current peak region, update the end index
        if potential_start <= end_index:
            end_index = potential_end
        else:
            # Integrate the previous peak region and update indices for the new peak
            peak_region = waveform[start_index:end_index]
            integration_result += np.trapz(peak_region)
            start_index = potential_start
            end_index = potential_end

    # Integrate the last peak region
    peak_region = waveform[start_index:end_index]
    integration_result += np.trapz(peak_region)

    return integration_result, properties

In [ ]:
def filter_metrics(
    metrics,
    min_score=0.5,
    max_score=np.inf,
    min_track_length=160,
    max_track_length=np.inf,
    max_tracks=1,
):
    print(f"min_score = {min_score}")
    print(f"max_score = {max_score}")
    print(f"min_track_length = {min_track_length}")
    print(f"max_track_length = {max_track_length}")
    print(f"max_tracks = {max_tracks}")

    filtered_metrics = {}

    for event_idx, metric in metrics.items():
        if len(metric) <= max_tracks + non_track_keys:
            candidate_metric = {
                track_idx: values
                for track_idx, values in metric.items()
                if isinstance(track_idx, str)
                or (
                    track_idx > 0
                    and values["RANSAC_score"] >= min_score
                    and values["RANSAC_score"] <= max_score
                    and values["Fit_norm"] >= min_track_length
                    and values["Fit_norm"] <= max_track_length
                )
            }
            if (
                len(candidate_metric) <= max_tracks + non_track_keys
                and len(candidate_metric) > non_track_keys
            ):
                filtered_metrics[event_idx] = candidate_metric

    print(f"{len(filtered_metrics)} metrics remaining")
    return filtered_metrics

## Uproot functions

In [ ]:
# Uproot
def load_charge(file_name, events=None):
    with uproot.open(file_name) as f:
        charge_df = f["HitTree"].arrays(library="pd").set_index("iev")
        if events is not None:
            charge_df = charge_df.loc[events]
    charge_df.rename({"eid": "event"}, axis=1, inplace=True)
    return charge_df

In [ ]:
def translate_coordinates(hit_df):
    hit_x = hit_df["hit_z"].apply(lambda x: [round(-i - x_translation, 1) for i in x])
    hit_y = hit_df["hit_y"].apply(lambda x: [round(i - y_translation, 1) for i in x])
    hit_z = hit_df["hit_x"].apply(lambda x: [round(i - z_translation, 1) for i in x])

    hit_df["hit_x"] = hit_x
    hit_df["hit_y"] = hit_y
    hit_df["hit_z"] = hit_z

    return hit_df

In [ ]:
def cut_volume(hit_df):
    additional_columns = [
        col for col in hit_df.columns if col not in ["hit_x", "hit_y", "hit_z"]
    ]
    columns_to_zip = ["hit_x", "hit_y", "hit_z"] + additional_columns

    def filter_hits(hit_row):
        filtered_hits = [
            hit
            for hit in zip(*[hit_row[col] for col in columns_to_zip])
            if all(
                abs(coord) <= detector / 2
                for coord, detector in zip(
                    hit[:3], (detector_x, detector_y, detector_z)
                )
            )
        ]
        return pd.Series(zip(*filtered_hits))

    hit_df[columns_to_zip] = hit_df.apply(filter_hits, axis=1)
    hit_df.dropna(subset=["hit_x", "hit_y", "hit_z"], inplace=True)

    return hit_df

In [ ]:
def cut_sipms(hit_df):
    additional_columns = [
        col for col in hit_df.columns if col not in ["hit_x", "hit_y", "hit_z"]
    ]
    columns_to_zip = ["hit_x", "hit_y", "hit_z"] + additional_columns

    def filter_hits(hit_row):
        filtered_hits = [
            hit
            for hit in zip(*[hit_row[col] for col in columns_to_zip])
            if not any(
                sipm_x - sipm_size / 2 <= hit[0] <= sipm_x + sipm_size / 2
                and sipm_y - sipm_size / 2 <= hit[1] <= sipm_y + sipm_size / 2
                for sipm_x, sipm_y in [
                    (
                        -detector_x / 2 + quadrant_size * (i + 0.5),
                        -detector_y / 2 + quadrant_size * (j + 0.5),
                    )
                    for i in range(detector_x // quadrant_size)
                    for j in range(detector_y // quadrant_size)
                ]
            )
        ]
        return pd.Series(zip(*filtered_hits))

    hit_df[columns_to_zip] = hit_df.apply(filter_hits, axis=1)
    hit_df.dropna(subset=["hit_x", "hit_y", "hit_z"], inplace=True)

    return hit_df

In [ ]:
def cut_chips(hit_df, first_chip=(2, 1)):
    additional_columns = [
        col for col in hit_df.columns if col not in ["hit_x", "hit_y", "hit_z"]
    ]
    columns_to_zip = ["hit_x", "hit_y", "hit_z"] + additional_columns

    def filter_hits(hit_row):
        filtered_hits = [
            hit
            for hit in zip(*[hit_row[col] for col in columns_to_zip])
            if not (
                any(
                    chip_x - quadrant_size / 2 <= hit[0] <= chip_x + quadrant_size / 2
                    and chip_y - quadrant_size / 2
                    <= hit[1]
                    <= chip_y + quadrant_size / 2
                    for chip_x, chip_y in [
                        (
                            -detector_x / 2
                            + quadrant_size * ((j - first_chip[1]) + 0.5),
                            detector_y / 2
                            - quadrant_size * ((i - first_chip[0]) + 0.5),
                        )
                        for (i, j) in [(3, 3), (4, 4), (5, 4), (6, 4)]
                    ]
                )
                or (
                    hit[0]
                    + detector_x / 2
                    - quadrant_size * ((2 - first_chip[1]))
                    + hit[1]
                    - detector_y / 2
                    + quadrant_size * ((4 - first_chip[0] + 1))
                    <= quadrant_size
                    and 0
                    <= hit[0] + detector_x / 2 - quadrant_size * ((2 - first_chip[1]))
                    <= quadrant_size
                    and 0
                    <= hit[1]
                    - detector_y / 2
                    + quadrant_size * ((4 - first_chip[0] + 1))
                    <= quadrant_size
                )
            )
        ]
        return pd.Series(zip(*filtered_hits))

    hit_df[columns_to_zip] = hit_df.apply(filter_hits, axis=1)
    hit_df.dropna(subset=["hit_x", "hit_y", "hit_z"], inplace=True)

    return hit_df

## Plotting

In [ ]:
class OOMFormatter(ScalarFormatter):
    def __init__(self, order=0, fformat="%1.1f", offset=True, mathText=True):
        self.oom = order
        self.fformat = fformat
        ScalarFormatter.__init__(self, useOffset=offset, useMathText=mathText)

    def _set_order_of_magnitude(self):
        self.orderOfMagnitude = self.oom

    def _set_format(self, vmin=None, vmax=None):
        self.format = self.fformat
        if self._useMathText:
            self.format = r"$\mathdefault{%s}$" % self.format


def set_common_ax_options(ax):
    ax.tick_params(axis="both", direction="inout", which="major", top=True, right=True)
    ax.set_axisbelow(True)
    ax.grid(alpha=0.25)
    if not ax.get_xscale() == "log":
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        if ax.get_xlim()[1] > 1.1:
            ax.xaxis.set_major_locator(MaxNLocator(integer=(ax.get_xlim()[1] > 2)))
            if ax.get_xlim()[1] > 1e3:
                ax.xaxis.set_major_formatter(OOMFormatter(3, "%1.1f"))

    if not ax.get_yscale() == "log":
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        if ax.get_ylim()[1] > 1.1:
            ax.yaxis.set_major_locator(MaxNLocator(integer=(ax.get_ylim()[1] > 2)))
            if ax.get_ylim()[1] > 1e3:
                ax.yaxis.set_major_formatter(OOMFormatter(3, "%1.1f"))

### Event display

In [ ]:
# Function to create a square based on center coordinates and side size
def create_square(center, side_size):
    x = [
        center[0] - side_size / 2,
        center[0] + side_size / 2,
        center[0] + side_size / 2,
        center[0] - side_size / 2,
        center[0] - side_size / 2,
    ]
    y = [
        center[1] - side_size / 2,
        center[1] - side_size / 2,
        center[1] + side_size / 2,
        center[1] + side_size / 2,
        center[1] - side_size / 2,
    ]
    z = [0, 0, 0, 0, 0]  # All z-coordinates are set to 0 to align with the xy-plane

    vertices = [(x[i], y[i], z[i]) for i in range(5)]
    square = [[vertices[0], vertices[1], vertices[2], vertices[3]]]
    return square

In [ ]:
def create_ed_axes(event_idx, charge, first_chip=(2, 1)):
    fig = plt.figure(figsize=(14, 6))
    ax3d = fig.add_subplot(121, projection="3d")
    ax2d = fig.add_subplot(122)
    fig.suptitle(f"Event {event_idx} - Charge = {charge} {q_unit}")
    grid_color = plt.rcParams["grid.color"]

    # Draw dead areas
    for i, j in [(3, 3), (4, 4), (5, 4), (6, 4), (4, 2)]:
        x = np.array([0, quadrant_size])
        y = -np.array([0, quadrant_size])
        ax2d.plot(
            np.power(-1, flip_x)
            * (x - detector_x / 2 + quadrant_size * (j - first_chip[1])),
            (y + detector_y / 2 - quadrant_size * (i - first_chip[0])),
            c=grid_color,
            lw=1,
        )
        if i == 4 and j == 2:
            x = np.array([quadrant_size, quadrant_size / 2])
            y = -np.array([quadrant_size / 2, 0])
        ax2d.plot(
            np.power(-1, flip_x)
            * (x[::-1] - detector_x / 2 + quadrant_size * (j - first_chip[1])),
            (y + detector_y / 2 - quadrant_size * (i - first_chip[0])),
            c=grid_color,
            lw=1,
        )

    # Adjust axes
    for ax in [ax3d, ax2d]:
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim([-detector_x / 2, detector_x / 2])
        ax.set_ylim([-detector_y / 2, detector_y / 2])
        ax.set_xlabel(f"x [{xy_unit}]")
        ax.set_ylabel(f"y [{xy_unit}]")
        ax.set_xticks(
            np.linspace(
                -detector_x / 2, detector_x / 2, (detector_x // quadrant_size) + 1
            )
        )
        ax.set_yticks(
            np.linspace(
                -detector_y / 2, detector_y / 2, (detector_y // quadrant_size) + 1
            )
        )
        ax.grid()

    ax2d.xaxis.set_minor_locator(AutoMinorLocator(8))
    ax2d.yaxis.set_minor_locator(AutoMinorLocator(8))
    ax2d.tick_params(axis="both", which="both", right=True, top=True)

    # Adjust z-axis
    ax3d.set_zlabel(f"z [{z_unit}]")
    # ax3d.zaxis.set_major_locator(MaxNLocator(integer=True))

    return fig, (ax2d, ax3d)


def event_display(
    event_idx,
    charge_df,
    plot_cyl=False,
    first_chip=(2, 1),
):
    if len(charge_df) < 2:
        return None

    # Plot the hits
    fig, axes = create_ed_axes(event_idx, round(sum(charge_df["q"])), first_chip)
    ax2d = axes[0]
    ax3d = axes[1]

    # Group by x and y coordinates and sum the z values
    unique_points, indices = np.unique(
        charge_df[["x", "y"]], axis=0, return_inverse=True
    )
    q_sum = np.bincount(indices, weights=charge_df["q"])

    # Plot the hits
    plot3d = ax3d.scatter(
        charge_df["x"],
        charge_df["y"],
        charge_df["z"],
        c=charge_df["q"],
        marker="s",
        s=round((30**4) / (detector_x * detector_y)),
        vmin=q_sum.min(),
        vmax=q_sum.max(),
    )
    plot2d = ax2d.scatter(
        unique_points[:, 0],
        unique_points[:, 1],
        c=q_sum,
        marker="s",
        s=round((30**4) / (detector_x * detector_y)),
        vmin=q_sum.min(),
        vmax=q_sum.max(),
    )
    cbar = plt.colorbar(plot2d)
    cbar.set_label(f"charge [{q_unit}]")

    # Cluster the hits
    labels = cluster_hits(charge_df[["x", "y", "z"]].to_numpy())

    # Fit clusters
    metrics = fit_hit_clusters(
        charge_df[["x", "y", "z"]].to_numpy(),
        charge_df["q"].to_numpy(),
        labels,
        ax2d,
        ax3d,
        plot_cyl,
    )

    # Draw missing SiPMs
    grid_color = plt.rcParams["grid.color"]

    # Draw SiPMs
    vertices_x = np.array([1, 1, -1, -1, 1]) * sipm_size / 2
    vertices_y = np.array([1, -1, -1, 1, 1]) * sipm_size / 2
    for missing_index in range(detector_x * detector_y // (quadrant_size**2)):
        col = (
            -detector_x / 2
            + quadrant_size / 2
            + (missing_index % (detector_x // quadrant_size)) * 32
        )
        row = (
            detector_y / 2
            - quadrant_size / 2
            - (missing_index // (detector_x // quadrant_size)) * 32
        )
        square = create_square((col, row), sipm_size)
        ax2d.fill(col + vertices_x, vertices_y + row, c=grid_color, zorder=5)
        ax3d.add_collection3d(Poly3DCollection(square, color=grid_color))

    ax3d.set_zlim([0, ax3d.get_zlim()[1]])
    # ax3d.view_init(160, 110, -85)
    ax3d.view_init(30, 20, 100)
    # ax3d.view_init(0, 0, 0)
    # ax3d.view_init(0, 0, 90)
    fig.tight_layout()

    if save_figures:
        os.makedirs(f"{file_label}/{event_idx}", exist_ok=True)
        fig.savefig(
            f"{file_label}/{event_idx}/event_{event_idx}.png",
            dpi=300,
            bbox_inches="tight",
        )

    return metrics

In [ ]:
def plot_fake_data(z_range, buffer=1, disable_coords=[(1, 2)], first_chip=(2, 1)):
    fake_data = generate_dead_area(z_range, buffer, disable_coords, first_chip)
    fake_x, fake_y, fake_z = fake_data[:, 0], fake_data[:, 1], fake_data[:, 2]

    fig = plt.figure()

    ax = fig.add_subplot(111)
    ax.set_aspect("equal", adjustable="box")
    ax.scatter(fake_x, fake_y, marker="s", s=round((20**4) / (detector_x * detector_y)))
    ax.set_xlim([-detector_x / 2, detector_x / 2])
    ax.set_ylim([-detector_y / 2, detector_y / 2])
    ax.set_xlabel(f"x [{xy_unit}]")
    ax.set_ylabel(f"y [{xy_unit}]")
    ax.set_xticks(
        np.linspace(-detector_x / 2, detector_x / 2, (detector_x // quadrant_size) + 1)
    )
    ax.set_yticks(
        np.linspace(-detector_y / 2, detector_y / 2, (detector_y // quadrant_size) + 1)
    )
    ax.xaxis.set_minor_locator(AutoMinorLocator(8))
    ax.yaxis.set_minor_locator(AutoMinorLocator(8))
    ax.grid()
    ax.tick_params(axis="both", which="both", top=True, right=True)
    fig.tight_layout()
    fig.savefig("fake_data_map.png", dpi=300, bbox_inches="tight")

### Tracks

In [ ]:
# Plot dQ versus X
def plot_dQ(dQ_array, event_idx, track_idx, dh, interpolate=False):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax_twinx = ax.twinx()

    fig.suptitle(
        rf"Event {event_idx} - Track {track_idx} - $dx = {round(dh,2)}$ {dh_unit}"
    )

    mean_dQ = np.mean(dQ_array[dQ_array > 0])
    non_zero_indices = np.where(dQ_array > 0)[0]

    # Check if there are non-zero values in dQ_array
    if non_zero_indices.size > 0:
        # Find the first non-zero index and add 2 indices before it
        first_index = max(non_zero_indices[0] - 2, 0)

        # Find the last non-zero index and add 2 indices after it
        last_index = min(non_zero_indices[-1] + 2, len(dQ_array))

        new_dQ_array = dQ_array.copy()[first_index:last_index]

        if interpolate:
            new_dQ_array[1:-1] = np.where(
                new_dQ_array[1:-1] == 0,
                mean_dQ,
                new_dQ_array[1:-1],
            )

        dQ_array = new_dQ_array

    ax.axhline(
        mean_dQ / dh,
        ls="--",
        c="red",
        label=rf"Mean = ${round(mean_dQ/dh,2)}$ {q_unit} {dh_unit}$^{{-1}}$",
        lw=1,
    )
    x_range = np.arange(0, len(dQ_array) * dh, dh)[: len(dQ_array)]

    ax.step(x_range, dQ_array / dh, where="mid")
    ax.set_xlabel(rf"$x$ [{dh_unit}]")
    ax.set_ylabel(rf"$dQ/dx$ [{q_unit} {dh_unit}$^{{-1}}$]")

    ax_twinx.step(x_range, np.cumsum(dQ_array), color="C1", where="mid")
    ax_twinx.set_ylabel(f"Q [{q_unit}]")

    for axes in [ax, ax_twinx]:
        set_common_ax_options(axes)

    h1, l1 = ax.get_legend_handles_labels()
    ax_twinx.legend(h1, l1, loc="lower center")

    ax.legend(loc="lower center")

    fig.tight_layout()
    if save_figures:
        os.makedirs(f"{file_label}/{event_idx}", exist_ok=True)
        fig.savefig(
            f"{file_label}/{event_idx}/dQ_E{event_idx}_T{track_idx}_{round(dh,2)}.png",
            dpi=300,
            bbox_inches="tight",
        )

In [ ]:
def plot_track_stats(
    metrics,
    limit_xrange=True,
    min_score=0.5,
    empty_ratio_lims=(0, 1),
    min_entries=2,
    lognorm=True,
    profile=False,
    bins=[40, 40],
):
    track_mean_dQdx = []
    track_std_dQdx = []
    track_length = []
    track_score = []
    track_z = []
    dQdx_list = []

    empty_count = 0
    short_count = 0
    for entry in metrics.values():
        for track, values in entry.items():
            if isinstance(track, str) or track <= 0:
                continue

            dQ = values["dQ"]
            dx = values["dx"]
            non_zero_mask = np.where(dQ > 0)[0]

            if len(dQ[non_zero_mask]) < min_entries:
                short_count += 1
                continue

            empty_ratio = sum(dQ[non_zero_mask[0] : non_zero_mask[-1] + 1] == 0) / (
                non_zero_mask[-1] - non_zero_mask[0] + 1
            )

            if empty_ratio > empty_ratio_lims[1] or empty_ratio < empty_ratio_lims[0]:
                empty_count += 1
                continue

            dQdx = dQ[non_zero_mask[0] : non_zero_mask[-1] + 1] / dx
            x_range = np.arange(0, len(dQdx) * dx, dx)[: len(dQdx)]
            dQdx_list.append(pd.Series(dQdx, index=x_range))

            non_zero_mask = non_zero_mask[non_zero_mask[0] : non_zero_mask[-1] + 1]
            track_mean_dQdx.append(np.mean(dQdx[dQdx > 0]))
            track_std_dQdx.append(np.std(dQdx[dQdx > 0]))
            track_length.append(values["Fit_norm"])
            track_score.append(values["RANSAC_score"])
            track_z.append(values["Fit_line"].point[2])

    print(f"Tracks with dead area outside {empty_ratio_lims} interval: {empty_count}")
    print(f"Tracks with less than {min_entries} entries: {short_count}")

    track_mean_dQdx = pd.Series(track_mean_dQdx)
    track_std_dQdx = pd.Series(track_std_dQdx)
    track_length = pd.Series(track_length)
    track_score = pd.Series(track_score)
    track_z = pd.Series(track_z)
    mask = (
        track_mean_dQdx.notna()
        * track_length.notna()
        * track_score.notna()
        * track_z.notna()
    )

    print(f"Remaining tracks: {sum(mask)}")

    track_mean_dQdx = track_mean_dQdx[mask]
    track_std_dQdx = track_std_dQdx[mask]
    track_length = track_length[mask]
    track_score = track_score[mask]
    track_z = track_z[mask]
    track_cv_dQdx = track_std_dQdx / track_mean_dQdx
    dQdx_list = [series for i, series in enumerate(dQdx_list) if mask[i]]

    score_mask = (track_score >= min_score).to_numpy()
    score_bool = (1 - score_mask).sum() > 0

    print(f"Tracks with score < {min_score}: {sum(mask)-sum(score_mask)}")

    print(f"Remaining_tracks: {sum(score_mask)}")

    # 1D histograms
    fig1 = plt.figure(figsize=(14, 6))

    ax11 = fig1.add_subplot(121)
    ax12 = fig1.add_subplot(122)

    n_all11, bins_all11, patches_all11 = ax11.hist(
        track_mean_dQdx, bins=bins[0], label="All tracks"
    )

    n_all12, bins_all12, patches_all12 = ax12.hist(
        track_length, bins=bins[0], label="All tracks"
    )

    if score_bool:
        n11, edges11, patches11 = ax11.hist(
            track_mean_dQdx[score_mask],
            bins=bins_all11,
            label=rf"Score $\geq {min_score}$",
        )
        ax12.hist(
            track_length[score_mask],
            bins=bins_all12,
            label=rf"Score $\geq {min_score}$",
        )

    ax11.set_xlabel(rf"Mean $dQ/dx$ [{q_unit} {dh_unit}$^{{-1}}$]")
    ax11.set_title(f"{len(track_mean_dQdx)} tracks")
    ax12.set_title(f"{len(track_length)} tracks")

    # 2D histograms
    def hist2d(x, y, ax, bins, lognorm, fit="Log", profile=False):
        if profile:
            hist, x_edges, y_edges = np.histogram2d(x, y, bins=bins)

            y_means = [
                np.mean(y[(x >= x_edges[i]) & (x < x_edges[i + 1])])
                for i in range(len(x_edges) - 1)
            ]
            y_stds = [
                np.std(y[(x >= x_edges[i]) & (x < x_edges[i + 1])])
                for i in range(len(x_edges) - 1)
            ]
            x_values = (x_edges[1:] + x_edges[:-1]) / 2
            bin_widths = [
                (x_edges[i + 1] - x_edges[i]) / 2 for i in range(len(x_edges) - 1)
            ]
            ax.errorbar(x_values, y_means, yerr=y_stds, xerr=bin_widths, fmt="o")

        else:
            hist2d = ax.hist2d(
                x,
                y,
                bins=bins,
                cmin=1,
                norm=LogNorm() if lognorm else None,
            )
        if fit == "Log":
            x_fit = np.log(x)
        elif fit == "Linear":
            x_fit = x
        else:
            return

        try:
            fit_p = np.polyfit(x_fit, y, 1)
        except:
            if fit == "Log":
                x_fit = x  # Try linear fit as a fallback
                fit = "Linear"
            elif fit == "Linear":
                x_fit = np.log(x)  # Try log fit as a fallback
                fit = "Log"
            try:
                fit_p = np.polyfit(x_fit, y, 1)
            except:
                return

        p = np.poly1d(fit_p)
        x_plot = np.arange(min(x), max(x), 1)

        if fit == "Log":
            y_plot = p(np.log(x_plot))
        else:
            y_plot = p(x_plot)

        ax.plot(x_plot, y_plot, c="salmon", ls="-", label=f"{fit} fit")

    fig2 = plt.figure(figsize=(14, 6))
    ax21 = fig2.add_subplot(121)
    ax22 = fig2.add_subplot(122)

    fig2.suptitle(f"{len(track_mean_dQdx)} tracks")
    ax21.set_ylabel(rf"Mean $dQ/dx$ [{q_unit} {dh_unit}$^{{-1}}$]")
    ax21.set_title("Mean dQ/dx vs. Track length")
    ax22.set_ylabel(rf"$dQ/dx$ CV")
    ax22.set_title("dQ/dx CV vs. Track length")

    hist2d21 = hist2d(
        track_length, track_mean_dQdx, ax21, bins, lognorm, fit="Log", profile=profile
    )

    hist2d22 = hist2d(
        track_length, track_cv_dQdx, ax22, bins, lognorm, fit="Linear", profile=profile
    )

    fig4 = plt.figure(figsize=(7, 6))
    ax4 = fig4.add_subplot(111)
    ax4.set_ylabel(f"Fit score")
    ax4.set_title("Fit score vs. Track length")

    hist2d4 = hist2d(
        track_length,
        track_score,
        ax4,
        [bins[0], 40],
        lognorm,
        fit="Log",
        profile=profile,
    )

    cut_dQdx_series = pd.concat(
        [series for i, series in enumerate(dQdx_list) if score_mask[i]]
    )
    cut_dQdx_series = cut_dQdx_series[cut_dQdx_series > 0].dropna().sort_index()
    dQdx_series = pd.concat(dQdx_list)
    dQdx_series = dQdx_series[dQdx_series > 0].dropna().sort_index()

    fig5 = plt.figure(figsize=(7 + 7 * score_bool, 6))
    ax51 = fig5.add_subplot(111 + 10 * score_bool)
    ax51.set_ylabel(rf"$dQ/dx$ [{q_unit} {dh_unit}$^{{-1}}$]")
    ax51.set_xlabel(rf"Residual range [{dh_unit}]")
    ax51.set_title(rf"{len(track_mean_dQdx)} tracks")

    hist2d(
        dQdx_series.index,
        dQdx_series,
        ax51,
        bins,
        lognorm,
        fit="Linear",
        profile=profile,
    )

    fig6 = plt.figure(figsize=(7 + 7 * score_bool, 6))
    ax61 = fig6.add_subplot(111 + 10 * score_bool)
    ax61.set_ylabel(rf"Mean $dQ/dx$ [{q_unit} {dh_unit}$^{{-1}}$]")
    ax61.set_xlabel(rf"Anode distance [{z_unit}]")
    ax61.set_title(rf"{len(track_z)} tracks")

    hist2d(track_z, track_mean_dQdx, ax61, bins, lognorm, fit="Linear", profile=profile)

    axes = [ax11, ax12, ax21, ax22, ax4, ax51, ax61]
    figs = [fig1, fig2, fig4, fig5, fig6]

    if score_bool:
        # 2D histograms after RANSAC score cut
        fig3 = plt.figure(figsize=(14, 6))
        ax31 = fig3.add_subplot(121)
        ax32 = fig3.add_subplot(122)
        ax31.set_ylabel(rf"Mean $dQ/dx$ [{q_unit} {dh_unit}$^{{-1}}$]")
        ax31.set_title(rf"Mean dQ/dx vs. Track length")
        ax32.set_ylabel(rf"$dQ/dx$ CV")
        ax32.set_title(rf"dQ/dx CV vs. Track length")
        fig3.suptitle(
            rf"Fit score $\geq {min_score}$ ({round(sum(score_mask)/len(score_mask)*100)}% of tracks)"
        )

        figs.append(fig3)
        axes.extend([ax31, ax32])

        hist2d31 = hist2d(
            track_length[score_mask],
            track_mean_dQdx[score_mask],
            ax31,
            bins,
            lognorm,
            fit="Log",
            profile=profile,
        )

        hist2d32 = hist2d(
            track_length[score_mask],
            track_cv_dQdx[score_mask],
            ax32,
            bins,
            lognorm,
            fit="Linear",
            profile=profile,
        )

        ax52 = fig5.add_subplot(122)
        axes.append(ax52)
        ax52.set_ylabel(rf"$dQ/dx$ [{q_unit} {dh_unit}$^{{-1}}$]")
        ax52.set_xlabel(rf"Residual range [{dh_unit}]")
        ax52.set_title(
            rf"Fit score $\geq {min_score}$ ({round(sum(score_mask)/len(score_mask)*100)}% of tracks)"
        )
        fig5.suptitle("dQ/dx vs. Residual range")

        hist2d(
            cut_dQdx_series.index,
            cut_dQdx_series,
            ax52,
            bins,
            lognorm,
            fit="Linear",
            profile=profile,
        )

        ax62 = fig6.add_subplot(122)
        axes.append(ax62)
        ax62.set_ylabel(rf"Mean $dQ/dx$ [{q_unit} {dh_unit}$^{{-1}}$]")
        ax62.set_xlabel(rf"Mean anode distance [{z_unit}]")
        ax62.set_title(
            rf"Fit score $\geq {min_score}$ ({round(sum(score_mask)/len(score_mask)*100)}% of tracks)"
        )
        fig6.suptitle("Mean dQ/dx vs. Mean anode distance")

        hist2d(
            track_z[score_mask],
            track_mean_dQdx[score_mask],
            ax62,
            bins,
            lognorm,
            fit="Linear",
            profile=profile,
        )

    max_track_legth = np.sqrt(detector_x**2 + detector_y**2 + detector_z**2)
    max_track_legth_xy = np.sqrt(detector_x**2 + detector_y**2)
    print("Max possible track length", round(max_track_legth, 2), "mm")
    print("Max possible track lengt on xy plane", round(max_track_legth_xy, 2), "mm")
    print("Max possible vertical track length", detector_y, "mm")

    for ax in axes:
        set_common_ax_options(ax)
        if ax == ax11 or ax == ax12:
            ax.set_ylabel("Counts")
        if ax != ax11:
            if not (
                ax == ax51 or ax == ax61 or (score_bool and (ax == ax52 or ax == ax62))
            ):
                ax.set_xlabel(f"Track length [{dh_unit}]")
            if max(track_length) > detector_y:
                ax.axvline(detector_y, c="g", ls="--", label="Max vertical length")
            if max(track_length) > max_track_legth_xy:
                ax.axvline(
                    max_track_legth_xy, c="orange", ls="--", label=r"Max length in $xy$"
                )
            if max(track_length) > max_track_legth:
                ax.axvline(max_track_legth, c="r", ls="--", label="Max length")

            if ax != ax12:
                if limit_xrange:
                    xlim = ax.get_xlim()
                    ax.set_xlim(xlim[0], min(max_track_legth + 10, xlim[1]))
                cbar = plt.colorbar(ax.collections[0])
                cbar.set_label("Counts" + (" [log]" if lognorm else ""))
        if not (not score_bool and ax == ax11):
            ax.legend(loc="lower right" if ax == ax4 else "upper right")

    for fig in figs:
        fig.tight_layout()

    if save_figures:
        entries = len(track_mean_dQdx)
        fig1.savefig(
            f"{file_label}/track_stats_1D_hist_{file_label}_{entries}.png",
            dpi=300,
            bbox_inches="tight",
        )
        fig2.savefig(
            f"{file_label}/track_stats_2D_hist_{file_label}_{entries}{'_profile' if profile else ''}.png",
            dpi=300,
            bbox_inches="tight",
        )
        fig4.savefig(
            f"{file_label}/track_stats_score_{file_label}_{entries}{'_profile' if profile else ''}.png",
            dpi=300,
            bbox_inches="tight",
        )
        fig5.savefig(
            f"{file_label}/track_stats_dQdx_{file_label}_{entries}{'_profile' if profile else ''}.png",
            dpi=300,
            bbox_inches="tight",
        )
        fig6.savefig(
            f"{file_label}/track_stats_dQdx_z_{file_label}_{entries}{'_profile' if profile else ''}.png",
            dpi=300,
            bbox_inches="tight",
        )
        if score_bool:
            fig3.savefig(
                f"{file_label}/track_stats_2D_hist_cut_{file_label}_{entries}{'_profile' if profile else ''}.png",
                dpi=300,
                bbox_inches="tight",
            )

# File handing

## Loading

In [ ]:
temp_filename = f"{file_label}/charge_df_{file_label}.bz2"
if not reload_files and os.path.isfile(temp_filename):
    charge_df = pd.read_csv(temp_filename, index_col=0)
    if not event_list is None:
        charge_df = charge_df.loc[event_list.intersection(charge_df.index)]
    # for column in :
    charge_df[charge_df.columns[9:]] = charge_df[charge_df.columns[9:]].applymap(
        lambda x: literal_eval(x) if isinstance(x, str) else x
    )
else:
    charge_df = load_charge(charge_file, events=event_list)

del temp_filename

In [ ]:
# Translate coordinates
if x_translation != 0 or y_translation != 0 or z_translation != 0:
    charge_df = translate_coordinates(charge_df)

In [ ]:
# Cut volume to live anode area
charge_df = cut_volume(charge_df)

In [ ]:
# Implement?
if simulate_dead_area:
    # Cut SiPMs from the anode
    charge_df = cut_sipms(charge_df)
    # Cut dead chips from anode
    charge_df = cut_chips(charge_df, first_chip=(2, 1) if detector_y == 160 else (1, 1))

In [ ]:
x, y, z = np.meshgrid(
    np.arange(-detector_x / 2, detector_x / 2, 1),
    np.arange(-detector_y / 2, detector_y / 2, 1),
    [1],
)

In [ ]:
temp = pd.DataFrame(
    {"hit_x": [x.flatten()], "hit_y": [y.flatten()], "hit_z": [z.flatten()]}
)

In [ ]:
row = cut_chips(temp, first_chip=(1, 1)).iloc[0]

In [ ]:
plt.scatter(row["hit_x"], row["hit_y"], s=1)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlim(-detector_x / 2, detector_x / 2)
plt.ylim(-detector_y / 2, detector_y / 2)
plt.xticks(
    np.linspace(-detector_x / 2, detector_x / 2, (detector_x // quadrant_size) + 1)
)
plt.yticks(
    np.linspace(-detector_y / 2, detector_y / 2, (detector_y // quadrant_size) + 1)
)
plt.grid()

## Saving

In [ ]:
os.makedirs(file_label, exist_ok=True)

In [ ]:
# Only save files if all events were considered, i.e. event_list is None
if event_list is None:
    charge_df.to_csv(f"{file_label}/charge_df_{file_label}.bz2")

## Verification

In [ ]:
charge_df.columns

In [ ]:
charge_df.size

### Histograms

#### Charge

In [ ]:
print(f"Hits charge in {q_unit}")
charge_df["hit_q"].apply(tuple).explode().hist()

In [ ]:
print(f"Hits x in {z_unit}")
charge_df["hit_x"].apply(tuple).explode().hist()

In [ ]:
print(f"Hits y in {z_unit}")
charge_df["hit_y"].apply(tuple).explode().hist()

In [ ]:
print(f"Hits z in {z_unit}")
charge_df["hit_z"].apply(tuple).explode().hist()

# Data fit

## Fake data map

In [ ]:
plot_fake_data(
    [np.arange(-detector_z, 0, 10)] if simulate_dead_area else [],
    first_chip=(2, 1) if detector_y == 160 else (1, 1),
    disable_coords=[(1, 2)] if simulate_dead_area else [],
)
if show_figures:
    plt.show()
else:
    plt.close("all")

## Main loop

In [ ]:
# Suppress the UndefinedMetricWarning
warnings.filterwarnings("ignore", category=Warning, module="sklearn")

In [ ]:
metrics = {}

if event_list is None:
    index_list = charge_df.index
else:
    index_list = charge_df.index.intersection(event_list)

for i, idx in enumerate(tqdm(index_list)):
    charge_values = pd.DataFrame(
        charge_df.loc[
            idx,
            [
                "hit_x",
                "hit_y",
                "hit_z",
                "hit_q",
            ],
        ].to_list(),
        index=["x", "y", "z", "q"],
    ).T

    # charge_values["q"] = charge_values["q"] * charge_gain  # Convert mV to ke

    if len(charge_values) > 2:
        if idx in individual_plots:
            metrics[idx] = event_display(
                idx,
                charge_values,
                plot_cyl=False,
                first_chip=(2, 1) if detector_y == 160 else (1, 1),
            )
            if show_figures:
                plt.show()
            else:
                plt.close()
        else:
            # Create a design matrix
            labels = cluster_hits(charge_values[["x", "y", "z"]].to_numpy())
            # Fit clusters
            metrics[idx] = fit_hit_clusters(
                charge_values[["x", "y", "z"]].to_numpy(),
                charge_values["q"].to_numpy(),
                labels,
            )

        # metrics[idx][
        #     "Pixel_mask"
        # ] = mask.to_numpy()  # Save masks to original dataframe for reference
        metrics[idx]["Total_charge"] = charge_values["q"].sum()

In [ ]:
# Reset the warning filter (optional)
warnings.filterwarnings("default", category=Warning)

## Metrics

In [ ]:
# Save the metrics to a pickle file
metrics_file = f"{file_label}/metrics_{file_label}.pkl"
if overwrite_metrics or not os.path.isfile(metrics_file):
    with open(metrics_file, "wb") as f:
        pickle.dump(metrics, f)

    print(f"Metrics saved to {metrics_file}")

In [ ]:
metrics